# MobileNetV2 - Road Damage Classification (RDD2020)
**Member:** IT22063564 · SE4050 Deep Learning 2026

Classifies road-damage crops into **D00** (longitudinal crack), **D10** (transverse crack), **D20** (alligator crack) and **D40** (pothole) with an ImageNet-pretrained MobileNetV2, using two-phase transfer learning.

All logic lives in `src/IT22063564/` - this notebook only runs it and shows the results:

| Step | Script |
|---|---|
| Build crops + image-level train/val/test split | `prepare_data.py` |
| Train (phase 1 head only, phase 2 fine-tune) | `train.py` |
| Final test evaluation | `evaluate.py` |

Settings: `src/IT22063564/configs/common.yaml` (shared: seed, split, image size) and `configs/mobilenetv2.yaml` (model).

**Runtime:** use a GPU. In Colab: *Runtime → Change runtime type → T4 GPU*.

## 1. Environment setup
On Colab this clones the repo (branch `feature/IT22063564`) and mounts Google Drive. Locally it just moves to the repo root.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DhanaM24/SE4050-Deep-Learning-Assignment.git"
BRANCH = "feature/IT22063564"
DRIVE_DIR = "/content/drive/MyDrive/data"      # Drive folder holding the zip
ZIP_NAME = "processed.zip"                    # crops built by prepare_data.py

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/SE4050-Deep-Learning-Assignment"):
        !git clone -b {BRANCH} {REPO_URL} /content/SE4050-Deep-Learning-Assignment
    os.chdir("/content/SE4050-Deep-Learning-Assignment")
    !pip install -q -r requirements.txt
else:
    # running locally from notebooks/IT22063564/
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

print("Working directory:", os.getcwd())

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none - training will be very slow")

## 2. Data
Two options:
* **A (recommended on Colab):** upload the already-built crops zip (~100 MB) to Drive (`MyDrive/data/processed.zip`) - it is unzipped automatically.
* **B:** extract `data/train.tar.gz` into `data/raw/` and run `prepare_data.py` to rebuild the crops.

The split is done **per source image** (70/15/15, seed 42) so no photo contributes crops to two splits. The RDD2020 `test1`/`test2` sets have no labels, so the test split comes from the training archive.

In [ ]:
from pathlib import Path

if not Path("data/processed/crops/train").exists():
    zip_path = f"{DRIVE_DIR}/{ZIP_NAME}" if IN_COLAB else "data/IT22063564_processed.zip"
    if Path(zip_path).exists():                     # option A
        !unzip -q -o "{zip_path}" -d data/_unzipped
        # accept zips whose top folder is "processed/", "crops/" or the split folders themselves
        crops = next(Path("data/_unzipped").rglob("train")).parent
        Path("data/processed").mkdir(parents=True, exist_ok=True)
        if crops.name == "crops":
            !cp -r "{crops.parent}/." data/processed/
        else:
            !mkdir -p data/processed/crops && cp -r "{crops}/." data/processed/crops/
        !rm -rf data/_unzipped
    else:                                           # option B
        !python src/IT22063564/prepare_data.py

print({s: sum(1 for _ in Path(f"data/processed/crops/{s}").rglob("*.jpg")) for s in ("train", "val", "test")})

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

classes = ["D00", "D10", "D20", "D40"]
counts = pd.DataFrame({c: [len(list(Path(f"data/processed/crops/{s}/{c}").glob("*.jpg"))) for s in ("train", "val", "test")]
                       for c in classes}, index=["train", "val", "test"])
display(counts)

counts.T.plot(kind="bar", figsize=(7, 4), title="Crops per class and split")
plt.ylabel("number of crops"); plt.xticks(rotation=0); plt.show()

## 3. Train
Phase 1: backbone frozen, head trained (lr 1e-3, ≤10 epochs). Phase 2: backbone layers 100+ unfrozen, BatchNorm frozen (lr 1e-5, ≤20 epochs). Early stopping on validation loss; the best checkpoint is saved to `results/models/mobilenetv2.keras`. Only train + val data are used here.

In [ ]:
!python src/IT22063564/train.py

In [ ]:
from IPython.display import Image, display
display(Image("results/figures/mobilenetv2_learning_curves.png"))
pd.read_csv("results/tables/mobilenetv2_history.csv").round(4)

## 4. Evaluate on the held-out test set
Run this **once**, after all tuning is finished.

In [ ]:
!python src/IT22063564/evaluate.py

In [ ]:
import json
metrics = json.load(open("results/tables/mobilenetv2_metrics.json"))
summary = pd.DataFrame({s: {k: v for k, v in metrics[s].items() if k != "confusion_matrix"}
                        for s in ("train", "val", "test")}).T
display(summary.round(4))
display(pd.Series(metrics["efficiency"], name="efficiency"))
display(pd.read_csv("results/tables/mobilenetv2_classification_report.csv", index_col=0).round(4))

In [ ]:
for fig in ("confusion_matrix", "roc_curves", "misclassified"):
    display(Image(f"results/figures/mobilenetv2_{fig}.png"))

## 5. Save results
Colab storage is temporary - copy results to Drive, then commit `results/tables` and `results/figures` to the repo (the `.keras` weights are git-ignored).

In [ ]:
if IN_COLAB:
    !mkdir -p "{DRIVE_DIR}/results" && cp -r results/* "{DRIVE_DIR}/results/"
    print("Copied to", DRIVE_DIR + "/results")